# Binary Variable Experiment

This notebook demonstrates the usage of the `conservative_pid` library to derive bounds for causal queries.
We consider a simple 2-binary variable setup (X, Y) where X causes Y.

In [1]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.append("src")

from symbolic import Query
from solver import LPSolver

from canonical import VectorizedCanonicalBasis
import sys
import os

import pandas as pd
import numpy as np
from symbolic import Variable, CounterfactualTerm, Event, P
from inference import ConservativePID

## 1. Setup Variables
We define two binary variables, X and Y, with domain {0, 1}.

In [2]:
X = Variable("X", (0, 1))
Y = Variable("Y", (0, 1))
variables = [X, Y]

## 2. Generate Synthetic Data
We generate data sampling from a known Bayesian Network with the following marginal and conditional probabilites:
- $P(X=1) = 0.5$
- $P(Y=1 | X=1) = 0.8$
- $P(Y=1 | X=0) = 0.3$

In [3]:
n_samples = 1000
np.random.seed(42)

x_data = np.random.binomial(1, 0.5, n_samples)

# Y depends on X
y_probs = np.where(x_data == 1, 0.8, 0.3)
y_data = np.random.binomial(1, y_probs, n_samples)

df = pd.DataFrame({"X": x_data, "Y": y_data})

## 3. Estimate Relative Frequencies
We compute the empirical joint distribution $\hat{P}(X, Y)$ from the data. 
This acts as the input `observational_data` for the inference engine.

In [26]:
# Compute counts
total = len(df)
observational_data = {}

for x_val in X.domain:
    for y_val in Y.domain:
        row_count = len(df[(df["X"] == x_val) & (df["Y"] == y_val)])
        observational_data[(x_val, y_val)] = row_count / total

print("Observational Data Distribution:", observational_data)

marginal_x = {
    x_val: sum(observational_data[(x_val, y_val)] for y_val in Y.domain)
    for x_val in X.domain
}
marginal_y = {
    y_val: sum(observational_data[(x_val, y_val)] for x_val in X.domain)
    for y_val in Y.domain
}

Observational Data Distribution: {(0, 0): 0.35, (0, 1): 0.153, (1, 0): 0.096, (1, 1): 0.401}


## 4. Inference with ConservativePID
We initialize the `ConservativePID` solver with the variables and the estimated distribution.

In [7]:
pid = ConservativePID(
    variables=variables,
    observational_probs=observational_data,
)
pid

ConservativePID(variables=[X, Y], data={(0, 0): 0.35, (0, 1): 0.153, (1, 0): 0.096, (1, 1): 0.401})

### A. Observational Query (Trivial)
Query: $P(Y=1, X=1)$

In [31]:
# Event: (Y=1 AND X=1)
q_obs: Query = P(Event({Y @ {}: 1}) & Event({X @ {}: 1}))
lb_obs, ub_obs = pid.infer(q_obs, causal_order=["X", "Y"])

print(f"Observational Bounds P(Y=1,  X=1): [{lb_obs:.4f}, {ub_obs:.4f}]")
print(f"Relative Frequency: {observational_data[(1, 1)]:.4f}")

2026-02-19 17:24:10.829 | SUCCESS  | inference:infer:18 - Input validation passed.
2026-02-19 17:24:10.831 | INFO     | inference:infer:20 - Using fixed causal order: ['X', 'Y']
2026-02-19 17:24:10.831 | DEBUG    | inference:_solve_for_order:6 - Solving for order: ['X', 'Y'] with monotonic=False
2026-02-19 17:24:10.835 | INFO     | src.canonical:__init__:44 - Generated Basis with 8 worlds.


Observational Bounds P(Y=1,  X=1): [0.4010, 0.4010]
Relative Frequency: 0.4010


Query: $P(Y=1 \mid X=1)$

In [36]:
q_obs = P(Y == 1, X == 1)
lb_obs, ub_obs = pid.infer(q_obs, causal_order=["X", "Y"])

print(f"Observational Bounds P(Y=1 | X=1): [{lb_obs:.4f}, {ub_obs:.4f}]")
print(f"Relative Frequency: {observational_data[(1, 1)] / marginal_x[1]:.4f}")

2026-02-19 17:26:27.582 | SUCCESS  | inference:infer:18 - Input validation passed.
2026-02-19 17:26:27.583 | INFO     | inference:infer:20 - Using fixed causal order: ['X', 'Y']
2026-02-19 17:26:27.584 | DEBUG    | inference:_solve_for_order:6 - Solving for order: ['X', 'Y'] with monotonic=False
2026-02-19 17:26:27.584 | INFO     | src.canonical:__init__:44 - Generated Basis with 8 worlds.


Observational Bounds P(Y=1 | X=1): [0.8068, 0.8068]
Relative Frequency: 0.8068


What happens if we reverse the order in this conditional query? $Y \to X$?

In [35]:
lb_obs, ub_obs = pid.infer(q_obs, causal_order=["Y", "X"])

print(f"Observational Bounds P(Y=1 | X=1): [{lb_obs:.4f}, {ub_obs:.4f}]")
print(f"Relative Frequency: {observational_data[(1, 1)] / marginal_x[1]:.4f}")

2026-02-19 17:26:25.290 | SUCCESS  | inference:infer:18 - Input validation passed.
2026-02-19 17:26:25.291 | INFO     | inference:infer:20 - Using fixed causal order: ['Y', 'X']
2026-02-19 17:26:25.291 | DEBUG    | inference:_solve_for_order:6 - Solving for order: ['Y', 'X'] with monotonic=False
2026-02-19 17:26:25.292 | INFO     | src.canonical:__init__:44 - Generated Basis with 8 worlds.


Observational Bounds P(Y=1 | X=1): [0.7238, 0.7238]
Relative Frequency: 0.8068


### B. Interventional Query
Query: $P(Y_{X=1} = 1)$
This represents the probability of Y=1 if we force X to be 1.

In [37]:
q_int = P(Y @ {X: 0} == 1)
lb_int, ub_int = pid.infer(q_int)

print(f"Interventional Bounds P(Y_{{X=1}} = 1): [{lb_int:.4f}, {ub_int:.4f}]")

2026-02-19 17:31:23.775 | SUCCESS  | inference:infer:18 - Input validation passed.
2026-02-19 17:31:23.776 | INFO     | inference:infer:31 - Found 1 valid causal orders compatible with the query.
2026-02-19 17:31:23.776 | DEBUG    | inference:_solve_for_order:6 - Solving for order: ['X', 'Y'] with monotonic=False
2026-02-19 17:31:23.776 | INFO     | src.canonical:__init__:44 - Generated Basis with 8 worlds.


Interventional Bounds P(Y_{X=1} = 1): [0.1530, 0.6500]


### C. Counterfactual Query
Query: $P(Y_{X=1} = 1 | X=0, Y=0)$
This asks: "Given that we observed X=0 and Y=0, what is the probability that Y would have been 1 if X had been 1?"

In [48]:
target = Event({Y @ {X: 1}: 1})
evidence = Event({X @ {}: 0, Y @ {}: 0})
q_cf = P(target, evidence)

lb_cf, ub_cf = pid.infer(q_cf)

print(f"Counterfactual Bounds P(Y_{{X=1}} = 1 | X=0, Y=0): [{lb_cf:.4f}, {ub_cf:.4f}]")

2026-02-19 17:32:42.100 | SUCCESS  | inference:infer:18 - Input validation passed.
2026-02-19 17:32:42.101 | INFO     | inference:infer:31 - Found 1 valid causal orders compatible with the query.
2026-02-19 17:32:42.101 | DEBUG    | inference:_solve_for_order:6 - Solving for order: ['X', 'Y'] with monotonic=False
2026-02-19 17:32:42.102 | INFO     | src.canonical:__init__:44 - Generated Basis with 8 worlds.


Counterfactual Bounds P(Y_{X=1} = 1 | X=0, Y=0): [0.0000, 1.0000]
